In [1]:
# 01 - Profile the raw data
#Goal: understand what is wrong with the raw transactions BEFORE writing any cleaning rules.
#Nothing in this notebook changes the data.

In [2]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
print("pandas", pd.__version__)
print("raw files:", [p.name for p in RAW_DIR.glob("*.csv")])

pandas 3.1.1
raw files: ['accounts.csv', 'transactions.csv']


In [3]:
tx = pd.read_csv(RAW_DIR / "transactions.csv", dtype=str, keep_default_na=False)
accounts = pd.read_csv(RAW_DIR / "accounts.csv", dtype=str, keep_default_na=False)

print("transactions:", tx.shape)   # (5206, 9)
print("accounts:", accounts.shape) # (300, 2)
tx.head()

transactions: (5206, 9)
accounts: (300, 2)


,txn_id,account_id,txn_ts,amount,currency,direction,channel,merchant,narrative
0,T000001087,ACC0000180,2026-11-02 20:21:39,1291.49,KES,CR,CARD,Java House,CARD payment Java House
1,T000001176,ACC0000107,2026-11-01 00:27:56,1948.45,KES,CR,ATM,KPLC Prepaid,ATM payment KPLC Prepaid
2,T000002423,ACC0000107,2026-11-05 22:31:41,468.42,KES,CR,MPESA,Jumia,MPESA payment Jumia
3,T000001352,ACC0000051,2026-11-01 06:07:53,1551.55,KES,CR,CARD,Safaricom Airtime,CARD payment Safaricom Airtime
4,T000004428,ACC0000084,2026-11-01 23:14:51,819.48,KES,CR,ATM,Naivas,ATM payment Naivas


In [5]:
(tx == "").sum()

txn_id         8
account_id     0
txn_ts         0
amount        15
currency       0
direction      0
channel        0
merchant       0
narrative      0
dtype: int64

In [6]:
ids = tx.loc[tx["txn_id"] != "", "txn_id"]   # ignore the blank ids
print("duplicate rows:", ids.duplicated().sum())   # 40

duplicate rows: 40


In [ ]:
amount = pd.to_numeric(tx["amount"], errors="coerce")   # anything that is not a number becomes NaN
print("not a number:", amount.isna().sum())  
print("zero or negative:", (amount <= 0).sum()) 

not a number: 25
zero or negative: 20


In [8]:
for col in ["currency", "channel", "direction"]:
    print(tx[col].value_counts(), "\n")
# currency: an unexpected 'XXX' (10)
# channel: 'FAX' (10) and ' mpesa ' (30)
# direction: '??' (10)

currency
KES    4780
USD     303
EUR     113
XXX      10
Name: count, dtype: int64 

channel
MPESA      2030
CARD       1302
ONLINE      804
ATM         774
BRANCH      256
 mpesa       30
FAX          10
Name: count, dtype: int64 

direction
CR    2650
DR    2546
??      10
Name: count, dtype: int64 



In [9]:
ts = pd.to_datetime(tx["txn_ts"], errors="coerce")
print("unparseable:", ts.isna().sum())                      # 8
print("in the future:", (ts > pd.Timestamp.now()).sum())    # 10
print("range:", ts.min(), "->", ts.max())

unparseable: 8
in the future: 10
range: 2026-11-01 00:02:59 -> 2026-12-31 10:00:00


In [10]:
orphans = ~tx["account_id"].isin(accounts["account_id"])
print("orphan accounts:", orphans.sum())   # 15

orphan accounts: 15


In [11]:
print("merchant with stray spaces:", (tx["merchant"] != tx["merchant"].str.strip()).sum())   # 20
print("narratives with a phone number:", tx["narrative"].str.contains(r"07\d{8}").sum())      # 407

merchant with stray spaces: 20
narratives with a phone number: 407


In [14]:
from IPython.display import display, Markdown

display(Markdown("""
## Findings → What We Do About Each

| Problem | Count | Action |
|---|---:|---|
| Duplicate txn_id | 40 | Reject, keep the first copy |
| Blank txn_id | 8 | Reject |
| Amount blank or not a number | 25 | Reject |
| Amount zero or negative | 20 | Reject |
| Unknown currency / channel / direction | 10 / 10 / 10 | Reject |
| Unparseable or future timestamp | 8 / 10 | Reject |
| Account not in accounts.csv | 15 | Reject |
| Channel with wrong case or spaces | 30 | Fix (trim + uppercase) |
| Merchant with stray spaces | 20 | Fix (trim) |
| Phone numbers in narrative | 407 | Mask before analytics |
"""))


## Findings → What We Do About Each

| Problem | Count | Action |
|---|---:|---|
| Duplicate txn_id | 40 | Reject, keep the first copy |
| Blank txn_id | 8 | Reject |
| Amount blank or not a number | 25 | Reject |
| Amount zero or negative | 20 | Reject |
| Unknown currency / channel / direction | 10 / 10 / 10 | Reject |
| Unparseable or future timestamp | 8 / 10 | Reject |
| Account not in accounts.csv | 15 | Reject |
| Channel with wrong case or spaces | 30 | Fix (trim + uppercase) |
| Merchant with stray spaces | 20 | Fix (trim) |
| Phone numbers in narrative | 407 | Mask before analytics |
